In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from getpass import getpass

# ----------------------------
# 1. Load model and tokenizer
# ----------------------------
model_name = "meta-llama/Llama-3.2-1B-Instruct"  # replace if needed

# ---- Hugging Face login ----
print("\nPlease paste your Hugging Face access token.")
hf_token = getpass("HF token: ")
login(token=hf_token)




Please paste your Hugging Face access token.
HF token: ··········


In [11]:
from tqdm.auto import tqdm
import re

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# ----------------------------
# 2. Example questions
# ----------------------------
# Replace this with your HC3 subset
# 1st 10 Reddit, then Finance, then Medicine, then Open QA, then Wiki CSAI
questions = [
    {"question_id": 0, "question": "Why is every book I hear about a NY Times # 1 Best Seller? ELI5 : Why is every book I hear about a NY Times # 1 Best Seller? Should n't there only be one # 1 best seller ? Please explain like I'm five."},
    {"question_id": 1, "question": "If salt is so bad for cars , why do we use it on the roads ? As the title states , why do we use it ? is there no other option or what ? Please explain like I'm five."},
    {"question_id": 2, "question": "Why do we still have SD TV channels when HD looks like SD on an old TV ? Could n't we just have the HD version of the channels & delete the SD ones ? Please explain like I'm five."},
    {"question_id": 3, "question": "Why has nobody assassinated Kim Jong - un He is such a pest and nuisance to basically the entire world except for China . Why has n't anyone had him assassinated yet ? Please explain like I'm five."},
    {"question_id": 4, "question": "How was airplane technology able to advance so quickly after the Wright Brothers ' first flight ? Mainly interested in how aviation was able to be deployed on a large scale during WWI . Please explain like I'm five."},
    {"question_id": 5, "question": "Why do humans have different colored eyes ? What causes / caused people to have different colors of eyes ? Is there a point to the color of your eyes other than superstitions ? Please explain like I'm five."},
    {"question_id": 6, "question": "Why I can not fabricate a religion that prevents me from going to school , then cite my first amendment rights when I am charged ? Prevents me from going to school by having , for example , supposed prayer times that coincide with school hours . Thanks ! Please explain like I'm five."},
    {"question_id": 7, "question": "What has changed that we frequently now throw away products instead of fixing them ? Is it the quality or nature of the products themselves , or more cultural in origin ? Please explain like I'm five."},
    {"question_id": 8, "question": "magic the gathering What is it . how popular is it . who plays it . The culture behind it .. etc . etc I have been hearing more and more about it , and after last nights South Park episode I became curious what it is .. Please explain like I'm five."},
    {"question_id": 9, "question": "What are prions and are they a big deal ? My AP Biology teacher mentioned them alongside viruses but he did n't say much else beyond that they are n't viruses and they deform protein structure . Please explain like I'm five."},
    {"question_id": 19141, "question": "Historical P/E ratios of small-cap vs. large-cap stocks?"},
    {"question_id": 19142, "question": "Should you co-sign a personal loan for a friend/family member? Why/why not?"},
    {"question_id": 19143, "question": "Should I avoid credit card use to improve our debt-to-income ratio?"},
    {"question_id": 19144, "question": "Difference between 'split and redemption' of shares and dividend"},
    {"question_id": 19145, "question": "Pros & cons of investing in gold vs. platinum?"},
    {"question_id": 19146, "question": "Pros and Cons of Interest Only Loans"},
    {"question_id": 19147, "question": "For a car loan, how much should I get preapproved for?"},
    {"question_id": 19148, "question": "Where can I lookup accurate current exchange rates for consumers?"},
    {"question_id": 19149, "question": "T-mobile stock: difference between TMUSP vs TMUS"},
    {"question_id": 19150, "question": "Money transfer from Australia to India - avoid receiving ends service tax"},
    {"question_id": 23074, "question": "Does Primolut N taken during pregnancy affect the baby?Hi doctor..I have taken primoult n tablet twice daily for 5 days to delay my periods in the month of march.I was not aware of the pregnancy at that time but now when I tested in April month it is positive in pregnancy.so my question here is does this tablet effect my baby"},
    {"question_id": 23075, "question": "Bloating and pain on right lower abdomen. Should i meet doctor?I have m any of the symptom of appendicitis.Bloatng pain on th right lower abdomen pressure on yhe left side brings about the pain ,to mention a few.It is 1'30 am where I am. should I try to reach my doctor or go to th er. I am 77 and otherwise in goodhealth WWW.WWWW.WW"},
    {"question_id": 23076, "question": "Is chest pain related to intake of clindamycin and oxycodone?Hi Dr. Bhatti, I was recently released from the hospital after a hand surgery and they provided me with Clindamycin 300mg and Oxycodone Acetaminophens. Ive taken this combination 3 times now and my chest feels really tight. Is there reason for me to worry?"},
    {"question_id": 23077, "question": "Q. Noticed a yellowish sag in the gums of my 13 months old baby. What is it?Hi doctor,My daughter is 13 months old. I just noticed big yellowish sag at the place of her first molar gum. I am much worried. Please help me."},
    {"question_id": 23078, "question": "Suggest remedy for low grade fever, hot and cold sweats with rashes on armsMy temp is 96.5,body aches,pain,abdomen tender,headache,tired and lazy,bla sick feeling not myself please help i know my body and no one will listen hody hasnt been quite right since bad iud infection in october also cold durin day sweat at night,rash on arm,cuts dont heal easy oh the pain i beg u please help"},
    {"question_id": 23079, "question": "Q. Kindly advise how to control wheezing and shortness of breath while having cold.Hello doctor,I have a wheezing problem and breathing shortness when I suffer from cold. I noticed I have dust allergy also. Day by day my problem is increasing. Now I have to take Levolin inhaler three times in a day. Is there any measure to control my problem?"},
    {"question_id": 23080, "question": "What are the side effects of changing the medication?I am cuurently and permanently on warfarin, and in one week s time I am to undergo surgery to have tor tendons repaired in my right shoulder. I have been directed to change temperarily from warfarin to glexane for six days prior to my surgery. I have been on warfarin for three years since a mechanical aortic valve was implanted during open heart surgery.What side effects should I expect? Thankyou"},
    {"question_id": 23081, "question": "What causes pressure and tightness in the chest region?Hello Doctor, Thanks for your time. I have been having some pressure / tightness in my chest region lately. I do not smoke or consume alcohol etc. I however sit for extended periods of time, as I am in IT industry. Could my occupation be causing this pain?"},
    {"question_id": 23082, "question": "Suggest treatment for folliculitis on penisHello, I was recently diagnosed as having folliculitis and it is showing up on my penis area. Sexual activity irritates it and therefore is becoming difficult. I am taking oral antibiotics and topical treatment but nothing is working. How can I treat this?"},
    {"question_id": 23083, "question": "Will the pain while peeing be due to kidney stone or UTI?I have a history with kidney stones and normally when i have one i get pain toward the end of urination in my clitorus. i Just got off of an antibiotic for a sinus infection and got a yeast infection and treated it with the 3 day monastat stuff. and now im feeling that pain again in my clit when i pee. is it just a kidney store or is it a UTI, also have some pain during intercorse"},
    {"question_id": 17112, "question": "what composer used sound mass"},
    {"question_id": 17113, "question": "where did the persian war take place"},
    {"question_id": 17114, "question": "what are add ons"},
    {"question_id": 17115, "question": "how does a dredge work?"},
    {"question_id": 17116, "question": "what classes are considered humanities"},
    {"question_id": 17117, "question": "where are colors on stoplight"},
    {"question_id": 17118, "question": "who is flo from progressive"},
    {"question_id": 17119, "question": "what happened to george o'malley on grey's anatomy?"},
    {"question_id": 17120, "question": "what makes of the united states"},
    {"question_id": 17121, "question": "what is atherosclerotic heart disease"},
    {"question_id": 18299, "question": "Please explain what is Animal cognition"},
    {"question_id": 18300, "question": "Please explain what is Human intelligence"},
    {"question_id": 18301, "question": "Please explain what is Oxford English Dictionary"},
    {"question_id": 18302, "question": "Please explain what is Oxford University Press"},
    {"question_id": 18303, "question": "Please explain what is AI applications"},
    {"question_id": 18304, "question": "Please explain what is Web search"},
    {"question_id": 18305, "question": "Please explain what is Google Search"},
    {"question_id": 18306, "question": "Please explain what is Recommender system"},
    {"question_id": 18307, "question": "Please explain what is Amazon (company)"},
    {"question_id": 18308, "question": "Please explain what is Natural-language understanding"},


]

df_questions = pd.DataFrame(questions)

# ----------------------------
# 3. Prompt templates
# ----------------------------
prompt_templates = {
    "normal": (
        "Write a short and clear answer (2-4 sentences, under 100 words) to the following question. Do not use examples, analogies, lists, headings, or step-by-step explanations. Do not ask questions. Do not include names, greetings, or dialogue. Start directly with the answer. \n\n"
        "{question}"
    ),
    "informal": (
        "Write a short answer (2-4 sentences, under 100 words) to the following question in a casual, informal way, like you are talking to a friend. Do not use examples, analogies, lists, headings, or step-by-step explanations. Do not ask questions. Do not include names, greetings, or dialogue. Use natural variation in tone and sentence length. Start directly with the answer. \n\n"
        "{question}"
    ),
    "slang": (
        "Write a short answer (2-4 sentences, under 100 words) to the following question using casual language and some slang, but keep it understandable. Do not use examples, analogies, lists, headings, or step-by-step explanations. Do not ask questions. Do not include names, greetings, or dialogue. Avoid assistant-like tone. Start directly with the answer. \n\n"
        "{question}"
    ),
    "human_like": (
        "Write a short answer (2-4 sentences, under 100 words) to the following question like a real knowledgeable person online.  Do not use headings, bullet points, numbered steps, examples, analogies, or phrases like Imagine, Hey, Let's, Oh boy, Here is, Let me explain, or Step 1. Do not ask questions. Do not include names, greetings, or dialogue. Use natural variation in sentence length and tone. Do not sound like an assistant or teacher. Allow small informalities, but keep the answer understandable. Start directly with the answer. \n\n"
        "{question}"
    ),
}

# ----------------------------
# 4. Functions
# ----------------------------
def generate_answer(question: str, prompt_style: str, max_new_tokens: int = 256, retry: bool = True):
    prompt = prompt_templates[prompt_style].format(question=question)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        min_new_tokens=30,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    generated_ids = output_ids[0][prompt_len:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    answer = clean_output(answer)

    answer_tokens = len(generated_ids)
    over_256 = answer_tokens >= 256

    # retry once if clearly bad
    if retry and is_bad_output(answer, answer_tokens):
        return generate_answer(question, prompt_style, max_new_tokens=max_new_tokens, retry=False)

    return answer, answer_tokens, over_256




def clean_output(text: str) -> str:
    text = text.strip()

    # remove common leading junk
    text = re.sub(r'^(Answer\s*:\s*)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^(A\.\s*)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^(Q\.\s*)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^[\.\?\-\:\;\,!\s]+', '', text)

    # remove common meta-notes
    text = re.sub(r'^\(Note:.*?\)\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^\[.*?\]\s*', '', text)

    return text.strip()


def is_bad_output(text: str, token_count: int) -> bool:
    stripped = text.strip().lower()

    if token_count < 10:
        return True
    if stripped == "":
        return True
    if stripped in {"!", "!!", ".", "..", "?", "??", "-", "--"}:
        return True

    bad_starts = [
        "answer:",
        "question:",
        "note:",
        "here are the details",
        "let me know when you're ready",
        "in my response",
    ]
    if any(stripped.startswith(x) for x in bad_starts):
        return True

    return False

# ----------------------------
# 5. Run pilot experiment
# ----------------------------
results = []

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions), desc="Questions"):

    print(f"\n=== Question ID: {row['question_id']} ===")
    print(f"Question: {row['question'][:150]}...\n")

    for prompt_style in prompt_templates.keys():
        print(f"-> Prompt style: {prompt_style}")

        answer, token_count, over_256 = generate_answer(
            question=row["question"],
            prompt_style=prompt_style,
            max_new_tokens=256
        )

        print(f"Token count: {token_count} | Over 256: {over_256}")
        preview = answer if len(answer) <= 500 else answer[:500] + "..."
        print(f"Answer: {preview}")
        print("-" * 80)

        results.append({
            "question_id": row["question_id"],
            "question": row["question"],
            "prompt_style": prompt_style,
            "answer": answer,
            "token_count": token_count,
            "over_256": over_256,
            "bad_output": is_bad_output(answer, token_count),
            "model_name": model_name
        })

        # save progress (VERY IMPORTANT for long runs)
        pd.DataFrame(results).to_csv("llama_prompt_pilot_partial.csv", index=False)

df_results = pd.DataFrame(results)
df_results.to_csv("llama_prompt_pilot.csv", index=False)

print("\nDone.")


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Questions:   0%|          | 0/50 [00:00<?, ?it/s]


=== Question ID: 0 ===
Question: Why is every book I hear about a NY Times # 1 Best Seller? ELI5 : Why is every book I hear about a NY Times # 1 Best Seller? Should n't there only be ...

-> Prompt style: normal
Token count: 135 | Over 256: False
Answer: Imagine you have two friends who both love reading books, but they read different kinds of books. One friend reads fantasy novels all day, and the other friend reads romance novels all day. Both friends want to know what their favorite type of book is, so they go to a big store together and find out which ones are popular among readers. They look at the shelves and see that fantasy novels are really popular, while romance novels are even more popular! So, why don't we just get one set of popular...
--------------------------------------------------------------------------------
-> Prompt style: informal
Token count: 106 | Over 256: False
Answer: Imagine your favorite toy car racing down the track. If it's the fastest car on the track,